In [1]:
## READS MNQ-TICK from HGT-Export SC chartbook

In [2]:
import pandas as pd
import numpy as np
import datetime as dt 
from pathlib import Path
import time

In [3]:
# Configuration: Style preferences
#plt.style.use('ggplot') # Good default for readability
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [4]:
import os
os.getcwd()

'/home/vm/pt/hgt-rl/mnq-tick'

In [5]:
inFile = f'/mnt/d/SierraChart/data/EXPORT/MNQ_OHLC_3SEC.csv'
outFile= f'data/mnq-ohlc-raw-3sec.pqt'

print(inFile, outFile,)

/mnt/d/SierraChart/data/EXPORT/MNQ_OHLC_3SEC.csv data/mnq-ohlc-raw-3sec.pqt


In [6]:
df = pd.read_csv(inFile)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11021818 entries, 0 to 11021817
Data columns (total 13 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Date       str    
 1   Time       str    
 2   Open       float64
 3   High       float64
 4   Low        float64
 5   Last       float64
 6   Volume     int64  
 7   #ofTrades  int64  
 8   OHLCAvg    float64
 9   HLCAvg     float64
 10  HLAvg      float64
 11  BidVolume  int64  
 12  AskVolume  int64  
dtypes: float64(7), int64(4), str(2)
memory usage: 1.3 GB
None
       Date             Time     Open      High       Low      Last  Volume  #ofTrades   OHLCAvg    HLCAvg     HLAvg  BidVolume  AskVolume
0  2022-1-3  08:00:00.000000  16431.0  16431.00  16429.75  16430.25      55         55  16430.50  16430.33  16430.38         29         26
1  2022-1-3  08:00:03.000000  16430.0  16430.75  16429.00  16429.25      28         28  16429.75  16429.67  16429.88         19          9
2  2022-1-3  08:00:06.000000  16429.5  16430.00  16429.2

In [8]:
# expand stupid SC date like 2026-7-8 to 2026-07-08
parts = df['Date'].astype(str).str.split('-', expand=True)

year  = parts[0]
month = parts[1].str.zfill(2)
day   = parts[2].str.zfill(2)

df['Date_norm'] = year + '-' + month + '-' + day

# join date and time and convert to wall clock datetime64
df['timestamp'] = pd.to_datetime(
    df['Date_norm'] + ' ' + df['Time'].astype(str),
    utc=False,            # keep as wall clock, no timezone
    errors='raise'        # or 'coerce' if you want bad rows as NaT
)

# remove temp columns
df = df.drop(columns=['Date', 'Date_norm', 'Time'])

# make timestamp first column
col = df.pop('timestamp')
df.insert(0, 'timestamp', col)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11021818 entries, 0 to 11021817
Data columns (total 12 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   Open       float64       
 2   High       float64       
 3   Low        float64       
 4   Last       float64       
 5   Volume     int64         
 6   #ofTrades  int64         
 7   OHLCAvg    float64       
 8   HLCAvg     float64       
 9   HLAvg      float64       
 10  BidVolume  int64         
 11  AskVolume  int64         
dtypes: datetime64[us](1), float64(7), int64(4)
memory usage: 1009.1 MB
None
            timestamp     Open      High       Low      Last  Volume  #ofTrades   OHLCAvg    HLCAvg     HLAvg  BidVolume  AskVolume
0 2022-01-03 08:00:00  16431.0  16431.00  16429.75  16430.25      55         55  16430.50  16430.33  16430.38         29         26
1 2022-01-03 08:00:03  16430.0  16430.75  16429.00  16429.25      28         28  16429.75  16429.67  16429.88         19 

In [10]:
df.drop(columns=['Volume','#ofTrades','BidVolume','AskVolume'], inplace=True) 

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 11021818 entries, 0 to 11021817
Data columns (total 8 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   Open       float64       
 2   High       float64       
 3   Low        float64       
 4   Last       float64       
 5   OHLCAvg    float64       
 6   HLCAvg     float64       
 7   HLAvg      float64       
dtypes: datetime64[us](1), float64(7)
memory usage: 672.7 MB
None
            timestamp     Open      High       Low      Last   OHLCAvg    HLCAvg     HLAvg
0 2022-01-03 08:00:00  16431.0  16431.00  16429.75  16430.25  16430.50  16430.33  16430.38
1 2022-01-03 08:00:03  16430.0  16430.75  16429.00  16429.25  16429.75  16429.67  16429.88
2 2022-01-03 08:00:06  16429.5  16430.00  16429.25  16429.25  16429.50  16429.50  16429.63
3 2022-01-03 08:00:09  16429.0  16430.00  16428.75  16429.50  16429.31  16429.42  16429.38
4 2022-01-03 08:00:12  16429.5  16429.75  16429.25  16429.25  1642

In [11]:
print(f'monotonic: {df["timestamp"].is_monotonic_increasing}')

df['date'] = df['timestamp'].dt.normalize()

day = pd.Timestamp("2025-11-28")
df = df[df["date"] != day]

df = df[df['timestamp'].dt.time >= dt.time(9, 30, 0)]

print(df.info())
print(df.head())

monotonic: True
<class 'pandas.DataFrame'>
Index: 8931784 entries, 1738 to 11021817
Data columns (total 9 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   Open       float64       
 2   High       float64       
 3   Low        float64       
 4   Last       float64       
 5   OHLCAvg    float64       
 6   HLCAvg     float64       
 7   HLAvg      float64       
 8   date       datetime64[us]
dtypes: datetime64[us](2), float64(7)
memory usage: 681.4 MB
None
               timestamp      Open      High       Low      Last   OHLCAvg    HLCAvg     HLAvg       date
1738 2022-01-03 09:30:00  16382.75  16388.25  16378.50  16388.00  16384.38  16384.92  16383.38 2022-01-03
1739 2022-01-03 09:30:03  16388.25  16395.25  16387.75  16395.25  16391.63  16392.75  16391.50 2022-01-03
1740 2022-01-03 09:30:06  16395.25  16397.25  16391.25  16394.00  16394.44  16394.17  16394.25 2022-01-03
1741 2022-01-03 09:30:09  16394.00  16396.50  16390.5

In [12]:
df.to_parquet(outFile, index=False)